# 🚀 VIENEU-TTS V3 TURBO - GOOGLE COLAB T4 GPU WORKER
Notebook này kết nối Google Colab T4 GPU với **TTS Proxy Gateway** qua Reverse WebSocket.
- Tự động Mount Google Drive và Cache Model 2GB để **không phải tải lại** mỗi lần bật lại Colab.
- Hỗ trợ nạp trực tiếp core engine từ repository: `https://github.com/pnnbao97/VieNeu-TTS.git`
- Nhận tác vụ theo thời gian thực và đẩy file trực tiếp lên Supabase Bucket `tts-audios`.

In [ ]:
# BƯỚC 1: KIỂM TRA GPU NVIDIA T4
import os
import torch

os.system("nvidia-smi")
if torch.cuda.is_available():
    print(f"✅ Đã phát hiện GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Chưa phát hiện GPU CUDA, vui lòng kiểm tra Runtime Type -> Chọn GPU T4!")

In [ ]:
# BƯỚC 2: MOUNT GOOGLE DRIVE ĐỂ LƯU MODEL & CACHE VĨNH VIỄN
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    CACHE_DIR = '/content/drive/MyDrive/VieNeu_TTS_Cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
    os.environ['TORCH_HOME'] = CACHE_DIR
    print(f"✅ Thư mục Cache trên Google Drive: {CACHE_DIR}")
except Exception as e:
    print(f"Thông báo khi mount Drive: {e}")

In [ ]:
# BƯỚC 3: TẢI REPOSITORY & CÀI ĐẶT THƯ VIỆN
import os
import sys
import subprocess

# 1. Đồng bộ mã nguồn TTS-Proxy-Platform từ GitHub vào Colab nếu chưa có
if not os.path.exists('/content/TTS-Proxy-Platform') and not os.path.exists('worker.py'):
    print("Đang tải mã nguồn worker từ GitHub...")
    subprocess.run(["git", "clone", "https://github.com/namvip55/TTS-Proxy-Platform.git", "/content/TTS-Proxy-Platform"], check=False)

# 2. Clone repository VieNeu-TTS nếu chưa có
if not os.path.exists('/content/VieNeu-TTS') and not os.path.exists('/content/drive/MyDrive/VieNeu-TTS'):
    print("Đang clone VieNeu-TTS từ GitHub...")
    subprocess.run(["git", "clone", "https://github.com/pnnbao97/VieNeu-TTS.git", "/content/VieNeu-TTS"], check=False)

# 3. Cài đặt các gói phụ thuộc qua pip
print("Đang cài đặt các thư viện cần thiết...")
pkgs = ["vieneu", "websockets", "supabase", "soundfile", "pydub", "python-dotenv", "accelerate", "transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅ Cài đặt môi trường VieNeu-TTS thành công!")

In [ ]:
# BƯỚC 4: THIẾT LẬP THÔNG SỐ KẾT NỐI GATEWAY VÀ SUPABASE
import os

# Điền thông số của bạn vào dưới đây:
os.environ['GATEWAY_URL'] = 'wss://your-backend-gateway.com'      # Địa chỉ Backend FastAPI Gateway
os.environ['WORKER_SECRET_KEY'] = 'vieneu_worker_secret_2026'     # Khóa bí mật Worker (khớp với backend/.env)
os.environ['SUPABASE_URL'] = 'https://your-project.supabase.co'    # URL Supabase Project
os.environ['SUPABASE_KEY'] = 'YOUR_SUPABASE_SERVICE_ROLE_KEY'     # Supabase Service Role Key (Điền khi chạy trên Colab)
os.environ['SUPABASE_BUCKET'] = 'tts-audios'

print("✅ Cấu hình môi trường thành công! Sẵn sàng khởi động Worker.")

In [ ]:
# BƯỚC 5: KHỞI CHẠY WORKER LẮNG NGHE JOB THEO THỜI GIAN THỰC
import os
import sys

# Tìm và chuyển đường dẫn vào thư mục chứa worker.py
search_paths = [
    ".",
    "/content/TTS-Proxy-Platform/colab-worker",
    "/content/colab-worker",
    "colab-worker"
]
found_dir = None
for p in search_paths:
    if os.path.exists(os.path.join(p, "worker.py")):
        found_dir = p
        break

if found_dir:
    os.chdir(found_dir)
    if found_dir not in sys.path:
        sys.path.insert(0, found_dir)
    print(f"✅ Đã chuyển vào thư mục: {os.getcwd()}")
    print(">>> Đang khởi động VieNeu Worker... <<<")
    # Chạy worker trực tiếp để hiển thị log theo thời gian thực
    import asyncio
    from worker import worker_loop
    await worker_loop()
else:
    print("❌ Không tìm thấy tệp worker.py! Vui lòng chạy lại BƯỚC 3 để tải mã nguồn từ GitHub.")